<a href="https://colab.research.google.com/github/GLzanetti/Projetos-de-Machine-Learning/blob/main/Aula7_MachineLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 7 - Machine Learning, prever a satisfação de clientes de um restaurante

## Análise de Satisfação de Clientes em Restaurante usando Machine Learning

Este projeto consiste no desenvolvimento e avaliação de modelos de Machine Learning para prever o nível de satisfação de clientes de um restaurante com base em suas escolhas e experiências (prato consumido, valor gasto e se o atendimento foi satisfatório ou não).

### 📋 Objetivo do Modelo
O objetivo principal é classificar se um cliente ficará **Satisfeito** ou **Insatisfeito** a partir de novas experiências de consumo. Com isso, o restaurante pode entender melhor o comportamento do seu público e identificar quais fatores (como o valor gasto ou a qualidade do atendimento) mais influenciam na experiência final do cliente.

### 🛠️ Detalhes da Implementação
Para resolver este problema de classificação binária, foram seguidas as seguintes etapas:
1. **Pré-processamento de Dados:** Tratamento de valores ausentes (remoção de registros sem rótulo de satisfação) e codificação de variáveis categóricas (`Prato`, `Atendimento` e `Satisfação`) em valores numéricos mapeados.
2. **Divisão de Dados:** Separação do dataset original em conjuntos de treino e teste utilizando a proporção clássica de 70/30.
3. **Modelagem:**
   * **Random Forest Classifier:** Um modelo baseado em árvores de decisão que apresentou performance máxima nos dados de teste.
   * **Logistic Regression:** Um modelo linear simples e eficiente, utilizado para comparação de métricas.
4. **Validação Cruzada (Cross-Validation):** Aplicação de validação cruzada com 5 partições ($k=5$) para avaliar a consistência e generalização de ambos os modelos em todo o conjunto de dados, evitando problemas de overfitting.
5. **Métricas de Avaliação:** Análise comparativa através de Acurácia, Precisão, Recall, F1-Score e Matriz de Confusão.
6. **Predição de Novos Casos:** Simulação com novos perfis de clientes para validar o comportamento prático dos modelos treinados.

In [ ]:
#Importação das bibliotecas

import pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)


In [ ]:
#Modelo de Random Forest
print("===================== Modelo Random Forest =====================")
#Importar o dataset
dados = pd.read_csv('dataset.csv')

#Remover linhas que não tem satisfação
dados = dados.dropna(subset=['Satisfação']).copy()

#Retirando nomes dos clientes da tabela, para não deixar enviesado
dados = dados.drop(columns=['Nome do cliente'])

#Mapeando tipos de comida, atendimento e satisfação em números
tipos_de_comida = {"Carne":1, "Frango":2, "Lasanha":3}
atendimento = {"Sim":1, "Não":2}
satisfacao = {"Insatisfeito": 0, "Satisfeito": 1}

#Transformando os dados do dataset nos números definidos
dados['Prato'] = dados['Prato'].map(tipos_de_comida)
dados['Atendimento'] = dados['Atendimento'].map(atendimento)
dados['Satisfação'] = dados['Satisfação'].map(satisfacao)

#Recolhendo os dados de x e y da base de dados
x = dados[['Prato', 'Gasto', 'Atendimento']]
y = dados['Satisfação']

#Separação entre treino e teste (70/30)
x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size=0.3, random_state=42)

#Instanciando o modelo
modelo = RandomForestClassifier(n_estimators=100,random_state=42)
modelo.fit(x_treino, y_treino) #Treinamento do modelo

#Avaliação do modelo
y_pred = modelo.predict(x_teste)

acuracia = accuracy_score(y_teste, y_pred) #Calcular acuracia
precisao = precision_score(y_teste, y_pred) #Calcular precisão
recall = recall_score(y_teste, y_pred) #Calcular recall
f1 = f1_score(y_teste, y_pred) #Calcular F1-Score
matriz_confusao = confusion_matrix(y_teste, y_pred) #Fazer a matriz de confusão

print("Acuracia: ", acuracia)
print("Precisão: ",precisao)
print("Recall: ",recall)
print("F1-Score: ",f1)
print("Matriz de Confusão: ")
print(matriz_confusao)

#Fazendo nova predição com dados novos
print("===================== Nova predição =====================")

#Novos dados para predição
predict = {1:{'Prato':'Lasanha', 'Gasto':51, 'Atendimento':'Sim'},
           2:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Sim'},
           3:{'Prato':'Carne', 'Gasto':63, 'Atendimento':'Sim'},
           4:{'Prato':'Lasanha', 'Gasto':40, 'Atendimento':'Não'},
           5:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Não'},
           6:{'Prato':'Carne', 'Gasto':100, 'Atendimento':'Não'}}

#Loop que passa por cada dicionário dentro do dicionário
for i in range(1,len(predict)+1):
  #Criando um dataframe para saber a predição do modelo
  predict_df =pd.DataFrame([[
      tipos_de_comida[predict[i]['Prato']],
      predict[i]['Gasto'],
      atendimento[predict[i]['Atendimento']]
                              ]], columns=x.columns)

  #Pegando a resposta do modelo em relação ao dataframe que criamos
  resposta_predict = modelo.predict(predict_df)[0]
  #Transformando os números em palavras de Satisfeito e Insatisfeito
  resultado = "Satisfeito" if resposta_predict == 1 else "Insatisfeito"

  print(f"Comeu {predict[i]['Prato']} e ficou {resultado}")

===================== Modelo Random Forest =====================
Acuracia:  1.0
Precisão:  1.0
Recall:  1.0
F1-Score:  1.0
Matriz de Confusão: 
[[ 72   0]
 [  0 116]]
===================== Nova predição =====================
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Satisfeito
Comeu Carne e ficou Satisfeito
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Insatisfeito
Comeu Carne e ficou Satisfeito


In [ ]:
#Modelo de Linear Regresion
print("===================== Modelo Logistc Regresion =====================")

#Importar o dataset
dados = pd.read_csv('dataset.csv')

#Remover linhas que sem satisfação
dados = dados.dropna(subset=['Satisfação']).copy()

#Retirando nomes dos clientes da tabela, para não deixar enviesado
dados = dados.drop(columns=['Nome do cliente'])

#Mapeando tipos de comida, atendimento e satisfação em números
tipos_de_comida = {"Carne":1, "Frango":2, "Lasanha":3}
atendimento = {"Sim":1, "Não":2}
satisfacao = {"Insatisfeito": 0, "Satisfeito": 1}

#Transformando os dados do dataset nos números definidos
dados['Prato'] = dados['Prato'].map(tipos_de_comida)
dados['Atendimento'] = dados['Atendimento'].map(atendimento)
dados['Satisfação'] = dados['Satisfação'].map(satisfacao)

#Recolhendo os dados de x e y da base de dados
x = dados[['Prato', 'Gasto', 'Atendimento']]
y = dados['Satisfação']

#Separação entre treino e teste (70/30)
x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, test_size=0.3, random_state=42)

#Instanciando o modelo
modelo = LogisticRegression()
modelo.fit(x_treino, y_treino) #Treinamento do modelo

#Avaliação do modelo
y_pred = modelo.predict(x_teste)

acuracia = accuracy_score(y_teste, y_pred) #Calcular acuracia
precisao = precision_score(y_teste, y_pred) #Calcular precisão
recall = recall_score(y_teste, y_pred) #Calcular recall
f1 = f1_score(y_teste, y_pred) #Calcular F1-Score
matriz_confusao = confusion_matrix(y_teste, y_pred) #Fazer a matriz de confusão

print("Acuracia: ", acuracia)
print("Precisão: ",precisao)
print("Recall: ",recall)
print("F1-Score: ",f1)
print("Matriz de Confusão: ")
print(matriz_confusao)

#Fazendo nova predição com dados novos
print("===================== Nova predição =====================")

#Novos dados para predição
predict = {1:{'Prato':'Lasanha', 'Gasto':51, 'Atendimento':'Sim'},
           2:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Sim'},
           3:{'Prato':'Carne', 'Gasto':63, 'Atendimento':'Sim'},
           4:{'Prato':'Lasanha', 'Gasto':40, 'Atendimento':'Não'},
           5:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Não'},
           6:{'Prato':'Carne', 'Gasto':100, 'Atendimento':'Não'}}

#Loop que passa por cada dicionário dentro do dicionário
for i in range(1,len(predict)+1):
  #Criando um dataframe para saber a predição do modelo
  predict_df =pd.DataFrame([[
      tipos_de_comida[predict[i]['Prato']],
      predict[i]['Gasto'],
      atendimento[predict[i]['Atendimento']]
                              ]], columns=x.columns)

  #Pegando a resposta do modelo em relação ao dataframe que criamos
  resposta_predict = modelo.predict(predict_df)[0]
  #Transformando os números em palavras de Satisfeito e Insatisfeito
  resultado = "Satisfeito" if resposta_predict == 1 else "Insatisfeito"

  print(f"Comeu {predict[i]['Prato']} e ficou {resultado}")

===================== Modelo Logistc Regresion =====================
Acuracia:  0.9893617021276596
Precisão:  1.0
Recall:  0.9827586206896551
F1-Score:  0.991304347826087
Matriz de Confusão: 
[[ 72   0]
 [  2 114]]
===================== Nova predição =====================
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Satisfeito
Comeu Carne e ficou Satisfeito
Comeu Lasanha e ficou Insatisfeito
Comeu Frango e ficou Insatisfeito
Comeu Carne e ficou Insatisfeito


In [ ]:
#Validação Cruzada
print("===================== Validação Cruzada (Random Forest) =====================")

#Importar o dataset
dados = pd.read_csv('dataset.csv')
dados.head()

#Remover linhas que sem satisfação
dados = dados.dropna(subset=['Satisfação']).copy()

#Retirando nomes dos clientes da tabela, para não deixar enviesado
dados = dados.drop(columns=['Nome do cliente'])

#Mapeando tipos de comida, atendimento e satisfação em números
tipos_de_comida = {"Carne":1, "Frango":2, "Lasanha":3}
atendimento = {"Sim":1, "Não":2}
satisfacao = {"Insatisfeito": 0, "Satisfeito": 1}

#Transformando os dados do dataset nos números definidos
dados['Prato'] = dados['Prato'].map(tipos_de_comida)
dados['Atendimento'] = dados['Atendimento'].map(atendimento)
dados['Satisfação'] = dados['Satisfação'].map(satisfacao)

#Recolhendo os dados de x e y da base de dados
x = dados[['Prato', 'Gasto', 'Atendimento']]
y = dados['Satisfação']

#Instanciando modelo
modelo_vc = RandomForestClassifier(n_estimators=100, random_state=42)

#Efetuando a validação cruzada com 5 partes
#Nessa parte o dataset será dividida em 5 partições e o modelo irá treinar
#5 vezes, sempre usando 4 partes para treinar e 1 para testar,
#e o resultado final vai ser a união das predições dos 5 testes.
y_pred_vc = cross_val_predict(modelo_vc, x, y, cv=5)

#Avaliação do modelo
#Aqui podemos descobrir a acuracia, precisão, recall, f1-score
#e criar a matriz de confusão do modelo, porem temos um diferencial
#onde não passamos somente o base de testes do dataset mas sim o dataset inteiro,
#por isso o atributo primário fica como y e não como y_teste
acuracia = accuracy_score(y, y_pred_vc) #Calcular acuracia
precisao = precision_score(y, y_pred_vc) #Calcular precisão
recall = recall_score(y, y_pred_vc) #Calcular recall
f1 = f1_score(y, y_pred_vc) #Calcular F1-Score
matriz_confusao = confusion_matrix(y, y_pred_vc) #Fazer a matriz de confusão

print("Acuracia: ", acuracia)
print("Precisão: ",precisao)
print("Recall: ",recall)
print("F1-Score: ",f1)
print("Matriz de Confusão: ")
print(matriz_confusao)

#Treinamento do modelo
modelo_vc = modelo_vc.fit(x, y)

#Fazendo nova predição com dados novos
print("===================== Nova predição =====================")

#Novos dados para predição
predict = {1:{'Prato':'Lasanha', 'Gasto':51, 'Atendimento':'Sim'},
           2:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Sim'},
           3:{'Prato':'Carne', 'Gasto':63, 'Atendimento':'Sim'},
           4:{'Prato':'Lasanha', 'Gasto':40, 'Atendimento':'Não'},
           5:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Não'},
           6:{'Prato':'Carne', 'Gasto':100, 'Atendimento':'Não'}}

#Loop que passa por cada dicionário dentro do dicionário
for i in range(1,len(predict)+1):
  #Criando um dataframe para saber a predição do modelo
  predict_df =pd.DataFrame([[
      tipos_de_comida[predict[i]['Prato']],
      predict[i]['Gasto'],
      atendimento[predict[i]['Atendimento']]
                              ]], columns=x.columns)

  #Pegando a resposta do modelo em relação ao dataframe que criamos
  resposta_predict = modelo_vc.predict(predict_df)[0]
  #Transformando os números em palavras de Satisfeito e Insatisfeito
  resultado = "Satisfeito" if resposta_predict == 1 else "Insatisfeito"

  print(f"Comeu {predict[i]['Prato']} e ficou {resultado}")

===================== Validação Cruzada (Random Forest) =====================
Acuracia:  1.0
Precisão:  1.0
Recall:  1.0
F1-Score:  1.0
Matriz de Confusão: 
[[240   0]
 [  0 386]]
===================== Nova predição =====================
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Satisfeito
Comeu Carne e ficou Satisfeito
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Insatisfeito
Comeu Carne e ficou Satisfeito


In [ ]:
#Validação Cruzada
print("===================== Validação Cruzada (Logistic Regression) =====================")

#Importar o dataset
dados = pd.read_csv('dataset.csv')
dados.head()

#Remover linhas que sem satisfação
dados = dados.dropna(subset=['Satisfação']).copy()

#Retirando nomes dos clientes da tabela, para não deixar enviesado
dados = dados.drop(columns=['Nome do cliente'])

#Mapeando tipos de comida, atendimento e satisfação em números
tipos_de_comida = {"Carne":1, "Frango":2, "Lasanha":3}
atendimento = {"Sim":1, "Não":2}
satisfacao = {"Insatisfeito": 0, "Satisfeito": 1}

#Transformando os dados do dataset nos números definidos
dados['Prato'] = dados['Prato'].map(tipos_de_comida)
dados['Atendimento'] = dados['Atendimento'].map(atendimento)
dados['Satisfação'] = dados['Satisfação'].map(satisfacao)

#Recolhendo os dados de x e y da base de dados
x = dados[['Prato', 'Gasto', 'Atendimento']]
y = dados['Satisfação']

#Instanciando modelo
modelo_vc = LogisticRegression()

#Efetuando a validação cruzada com 5 partes
#Nessa parte o dataset será dividida em 5 partições e o modelo irá treinar
#5 vezes, sempre usando 4 partes para treinar e 1 para testar,
#e o resultado final vai ser a união das predições dos 5 testes.
y_pred_vc = cross_val_predict(modelo_vc, x, y, cv=5)

#Avaliação do modelo
#Aqui podemos descobrir a acuracia, precisão, recall, f1-score
#e criar a matriz de confusão do modelo, porem temos um diferencial
#onde não passamos somente o base de testes do dataset mas sim o dataset inteiro,
#por isso o atributo primário fica como y e não como y_teste
acuracia = accuracy_score(y, y_pred_vc) #Calcular acuracia
precisao = precision_score(y, y_pred_vc) #Calcular precisão
recall = recall_score(y, y_pred_vc) #Calcular recall
f1 = f1_score(y, y_pred_vc) #Calcular F1-Score
matriz_confusao = confusion_matrix(y, y_pred_vc) #Fazer a matriz de confusão

print("Acuracia: ", acuracia)
print("Precisão: ",precisao)
print("Recall: ",recall)
print("F1-Score: ",f1)
print("Matriz de Confusão: ")
print(matriz_confusao)

#Treinamento do modelo
modelo_vc = modelo_vc.fit(x, y)

#Fazendo nova predição com dados novos
print("===================== Nova predição =====================")

#Novos dados para predição
predict = {1:{'Prato':'Lasanha', 'Gasto':51, 'Atendimento':'Sim'},
           2:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Sim'},
           3:{'Prato':'Carne', 'Gasto':63, 'Atendimento':'Sim'},
           4:{'Prato':'Lasanha', 'Gasto':40, 'Atendimento':'Não'},
           5:{'Prato':'Frango', 'Gasto':21, 'Atendimento':'Não'},
           6:{'Prato':'Carne', 'Gasto':100, 'Atendimento':'Não'}}

#Loop que passa por cada dicionário dentro do dicionário
for i in range(1,len(predict)+1):
  #Criando um dataframe para saber a predição do modelo
  predict_df =pd.DataFrame([[
      tipos_de_comida[predict[i]['Prato']],
      predict[i]['Gasto'],
      atendimento[predict[i]['Atendimento']]
                              ]], columns=x.columns)

  #Pegando a resposta do modelo em relação ao dataframe que criamos
  resposta_predict = modelo_vc.predict(predict_df)[0]
  #Transformando os números em palavras de Satisfeito e Insatisfeito
  resultado = "Satisfeito" if resposta_predict == 1 else "Insatisfeito"

  print(f"Comeu {predict[i]['Prato']} e ficou {resultado}")

===================== Validação Cruzada (Logistic Regression) =====================
Acuracia:  0.9792332268370607
Precisão:  1.0
Recall:  0.966321243523316
F1-Score:  0.9828722002635046
Matriz de Confusão: 
[[240   0]
 [ 13 373]]
===================== Nova predição =====================
Comeu Lasanha e ficou Satisfeito
Comeu Frango e ficou Satisfeito
Comeu Carne e ficou Satisfeito
Comeu Lasanha e ficou Insatisfeito
Comeu Frango e ficou Insatisfeito
Comeu Carne e ficou Insatisfeito
